# Fabric 03 — Silver Layer (with validation + rejected_records)
**Purpose:** mirror of Databricks 03_silver in Fabric. Demonstrates Spark portability.          
**Input:** `bronze_yellow_taxi`         
**Outputs:** `silver_yellow_taxi` (clean), `rejected_records` (quarantined with rule names)

In [1]:
import uuid
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

LOG_TABLE = "pipeline_log"   # <-- only line different from Databricks
RUN_ID = str(uuid.uuid4())
PIPELINE_NAME = "nyc_taxi_yellow"

LOG_SCHEMA = StructType([
    StructField("pipeline_name", StringType(),    nullable=False),
    StructField("run_id",        StringType(),    nullable=False),
    StructField("stage",         StringType(),    nullable=False),
    StructField("rows_in",       LongType(),      nullable=True),
    StructField("rows_out",      LongType(),      nullable=True),
    StructField("status",        StringType(),    nullable=False),
    StructField("error_message", StringType(),    nullable=True),
    StructField("run_timestamp", TimestampType(), nullable=False),
])

def log_pipeline_run(stage, rows_in, rows_out, status, error_message=None):
    log_row = spark.createDataFrame(
        [(PIPELINE_NAME, RUN_ID, stage, rows_in, rows_out, status, error_message, datetime.utcnow())],
        schema=LOG_SCHEMA
    )
    (log_row.write.format("delta").mode("append").saveAsTable(LOG_TABLE))
    print(f"[{stage}] {status} | rows_in={rows_in:,} rows_out={rows_out:,}")

StatementMeta(, 24fa6347-f4f8-480d-b620-a9f12aea1214, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F

BRONZE_TABLE   = "bronze_yellow_taxi"
SILVER_TABLE   = "silver_yellow_taxi"
REJECTED_TABLE = "rejected_records"

try:
    df_bronze = spark.table(BRONZE_TABLE)
    rows_in = df_bronze.count()

    rename_map = {
        "VendorID":             "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime":"dropoff_datetime",
        "RatecodeID":           "ratecode_id",
        "PULocationID":         "pickup_location_id",
        "DOLocationID":         "dropoff_location_id",
        "Airport_fee":          "airport_fee",
    }
    df = df_bronze
    for old, new in rename_map.items():
        if old in df.columns:
            df = df.withColumnRenamed(old, new)

    df = df.dropna(subset=["pickup_datetime", "dropoff_datetime", "fare_amount", "passenger_count"])

    df = (df
        .withColumn("trip_duration_minutes",
                    (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60.0)
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
    )

    df = df.withColumn(
        "validation_failures",
        F.array_compact(F.array(
            F.when(~(F.col("fare_amount") > 0),                                          F.lit("fare_not_positive")),
            F.when(~((F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 6)), F.lit("passenger_count_out_of_range")),
            F.when(~(F.col("trip_distance") >= 0),                                       F.lit("negative_distance")),
            F.when(~(F.col("pickup_datetime") < F.col("dropoff_datetime")),              F.lit("non_chronological_times")),
        ))
    )

    df_clean    = df.filter(F.size("validation_failures") == 0).drop("validation_failures")
    df_rejected = df.filter(F.size("validation_failures") >  0)

    (df_clean.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(SILVER_TABLE))
    rows_clean = spark.table(SILVER_TABLE).count()

    (df_rejected.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(REJECTED_TABLE))
    rows_rejected = spark.table(REJECTED_TABLE).count()

    log_pipeline_run("silver", rows_in, rows_clean, "SUCCESS")
    print(f"Quarantined {rows_rejected:,} rows to {REJECTED_TABLE}")

except Exception as e:
    log_pipeline_run("silver", -1, 0, "FAILED", str(e))
    raise

StatementMeta(, 24fa6347-f4f8-480d-b620-a9f12aea1214, 4, Finished, Available, Finished, False)

[silver] SUCCESS | rows_in=2,964,624 rows_out=2,756,127
Quarantined 68,335 rows to rejected_records


In [3]:
bronze_total = spark.table(BRONZE_TABLE).count()
silver_total = spark.table(SILVER_TABLE).count()
rejected_total = spark.table(REJECTED_TABLE).count()

print("=== Pipeline row reconciliation (Fabric) ===")
print(f"Bronze:                          {bronze_total:,}")
print(f"  - Dropped (structural nulls):  {bronze_total - silver_total - rejected_total:,}")
print(f"  - Quarantined (rule failures): {rejected_total:,}")
print(f"  = Silver (clean):              {silver_total:,}")

print("\n=== Rejection breakdown by rule ===")
spark.sql(f"""
    SELECT failure_rule, COUNT(*) AS row_count
    FROM (SELECT explode(validation_failures) AS failure_rule FROM {REJECTED_TABLE})
    GROUP BY failure_rule
    ORDER BY row_count DESC
""").show(truncate=False)

StatementMeta(, 24fa6347-f4f8-480d-b620-a9f12aea1214, 5, Finished, Available, Finished, False)

=== Pipeline row reconciliation (Fabric) ===
Bronze:                          2,964,624
  - Dropped (structural nulls):  140,162
  - Quarantined (rule failures): 68,335
  = Silver (clean):              2,756,127

=== Rejection breakdown by rule ===
+----------------------------+---------+
|failure_rule                |row_count|
+----------------------------+---------+
|fare_not_positive           |36225    |
|passenger_count_out_of_range|31525    |
|non_chronological_times     |759      |
+----------------------------+---------+

